# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing entities via their `@id` fields.

### Dataset Source
The dataset source is a Croissant schema accessible at:

[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'
url = croissant_url

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print metadata overview
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

print("Published Date:", getattr(metadata, 'datePublished', None))
print("Version:", getattr(metadata, 'version', None))
print("License:", getattr(metadata, 'license', None))


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Each entity (record set, field, column) is referenced by its `@id`.

Let's enumerate all record sets present in the dataset, along with their field and column IDs.


In [ ]:
# List all record sets in the dataset, referencing @id
record_sets = list(dataset.record_sets.keys())
print("Available record sets (@id):")
for rs_id in record_sets:
    print(f"- {rs_id}")
    fields = dataset.record_sets[rs_id].fields
    print("  Fields:")
    for field_id in fields:
        print(f"    - {field_id}")
        if hasattr(fields[field_id], 'columns'):
            columns = fields[field_id].columns
            if columns:
                print("      Columns:")
                for col in columns:
                    print(f"        - {col}")
    print()


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Here, we use each record set's `@id` as defined in the previous overview.


In [ ]:
# Extract data from each record set using their @id
dataframes = {}

# If record_sets is empty, explain why
if not record_sets:
    print("No record sets found in this dataset. Please refer to metadata or distribution files.")
else:
    for record_set_id in record_sets:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Extracted DataFrame for record set {record_set_id} with shape {df.shape}")

    # Pick the first record set for demonstration
    demo_record_set = record_sets[0]
    print("\nColumns in the DataFrame:")
    print(dataframes[demo_record_set].columns.tolist())
    print("\nPreview:")
    print(dataframes[demo_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing values, grouping data.

For demonstration, select a numeric field and a grouping field referencing their `@id`s.

In [ ]:
# --- EDA Example ---
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if not record_sets:
    print("EDA step skipped: No record sets or data available.")
else:
    df = dataframes[demo_record_set]
    # List available columns
    print("Columns available for EDA:", df.columns.tolist())

    # Attempt to select a numeric field for demonstration
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id:
        print(f"Numeric field selected (@id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if len(df) > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a field
        group_field_id = None
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable group field (@id) found for grouping.")
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Visualize the distribution or relationships between fields using columns referenced via `@id`.

In [ ]:
# Visualization Example
import matplotlib.pyplot as plt

if not record_sets or not dataframes:
    print("No data available for visualization.")
else:
    df = dataframes[demo_record_set]
    # Try to find a numeric field to visualize
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id:
        plt.figure(figsize=(8, 4))
        df[numeric_field_id].hist(bins=20)
        plt.title(f"Distribution of {numeric_field_id} (@id)")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated structured loading, exploration, and analysis of the FAIR^2 dataset using `mlcroissant`, referencing entities by their `@id` for clarity and reproducibility.

**Key Observations:**
- Dataset includes regression results across diverse socio-demographics and intervention variables.
- Missing data and overrepresentation biases exist, as described in metadata.
- Fields, columns, and grouping can be dynamically explored for policy analysis or research.

For further analysis, consult the Croissant metadata and the full schema available at the provided URL.